In [ ]:
# model.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import HeteroConv, GCNConv, SAGEConv, Linear

class DrugEncoder(nn.Module):
    """
    Heterogeneous encoder that computes embeddings for 'drug' nodes
    using relations: drug--binds-->protein and protein--interacts-->protein and drug--interacts-->drug.
    """
    def __init__(self, in_drug, in_protein, hidden_dim=256, out_dim=128):
        super().__init__()
        # two-layer HeteroConv
        conv1 = {
            ("drug", "interacts", "drug"): SAGEConv(in_drug, hidden_dim),
            ("drug", "binds", "protein"): SAGEConv(in_drug, hidden_dim),
            ("protein", "interacts", "protein"): SAGEConv(in_protein, hidden_dim),
            ("protein", "rev_binds", "drug"): SAGEConv(in_protein, hidden_dim)  # reverse relation
        }
        conv2 = {
            ("drug", "interacts", "drug"): SAGEConv(hidden_dim, out_dim),
            ("drug", "binds", "protein"): SAGEConv(hidden_dim, out_dim),
            ("protein", "interacts", "protein"): SAGEConv(hidden_dim, out_dim),
            ("protein", "rev_binds", "drug"): SAGEConv(hidden_dim, out_dim)
        }
        self.conv1 = HeteroConv(conv1, aggr="sum")
        self.conv2 = HeteroConv(conv2, aggr="sum")
        self.lin_drug = Linear(in_drug, hidden_dim)
        self.lin_prot = Linear(in_protein, hidden_dim)

    def forward(self, x_dict, edge_index_dict):
        # prepare reverse binds relation if exists
        if ("drug", "binds", "protein") in edge_index_dict:
            src, dst = edge_index_dict[("drug", "binds", "protein")]
            edge_rev = torch.stack([dst, src], dim=0)
            edge_index_dict[("protein", "rev_binds", "drug")] = edge_rev

        x_drug = self.lin_drug(x_dict["drug"])
        x_prot = self.lin_prot(x_dict["protein"])
        x1 = {"drug": x_drug, "protein": x_prot}
        x1 = self.conv1(x1, edge_index_dict)
        x1 = {k: F.relu(v) for k, v in x1.items()}
        x2 = self.conv2(x1, edge_index_dict)
        # Return the drug embeddings
        return x2["drug"]

class ComboPredictor(nn.Module):
    """
    Combines drug embeddings (by mean pooling) and predicts multi-label side-effects.
    """
    def __init__(self, drug_emb_dim, hidden_dim, num_sideeffects, dropout=0.2):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(drug_emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_sideeffects)
        )

    def forward(self, combo_drug_embeddings):
        # combo_drug_embeddings: (batch, k, emb_dim) OR a list of variable-length embeddings
        if isinstance(combo_drug_embeddings, list):
            # variable-length: mean each
            pooled = torch.stack([c.mean(dim=0) for c in combo_drug_embeddings], dim=0)
        else:
            # assume tensor (batch, k, emb)
            pooled = combo_drug_embeddings.mean(dim=1)
        return self.mlp(pooled)  # logits

class HODDIModel(nn.Module):
    def __init__(self, drug_in_dim, prot_in_dim, drug_hidden=256, drug_out=128, cls_hidden=256, num_sideeffects=1000):
        super().__init__()
        self.encoder = DrugEncoder(drug_in_dim, prot_in_dim, hidden_dim=drug_hidden, out_dim=drug_out)
        self.classifier = ComboPredictor(drug_out, cls_hidden, num_sideeffects)

    def forward(self, x_dict, edge_index_dict, combo_batch):
        # combo_batch: either list of lists of drug indices (per batch item), or (batch,k) tensor with padded -1s
        drug_emb = self.encoder(x_dict, edge_index_dict)  # (num_drugs, emb)
        # build combo embeddings
        if isinstance(combo_batch, torch.Tensor):
            # padded tensor with -1 for padding
            mask = (combo_batch >= 0)
            pad = combo_batch.clone()
            pad[pad < 0] = 0
            emb = drug_emb[pad]  # (batch,k,emb)
            emb = emb * mask.unsqueeze(-1).float()
            # to avoid zero-division, compute per-row counts
            counts = mask.sum(dim=1).clamp(min=1).unsqueeze(-1).float()
            emb = emb.sum(dim=1) / counts  # mean pooling
            logits = self.classifier.mlp(emb)
            return logits
        else:
            # list of variable-length index lists: convert to list of tensors
            emb_list = []
            for idxs in combo_batch:
                e = drug_emb[torch.tensor(idxs, dtype=torch.long, device=drug_emb.device)]
                emb_list.append(e)
            logits = self.classifier(emb_list)
            return logits
